# Activity: Real-time Crypto Currency Price Quotes with WebSockets

In this activity, we will connect to the Kraken cryptocurrency exchange using WebSockets to receive real-time price quotes for Bitcoin (BTC) and implement a message handler to process different types of updates.

__Why are we looking at this?__ WebSockets enable persistent, bidirectional communication between clients and servers, making them ideal for applications requiring real-time data updates. Unlike traditional HTTP requests that require repeated polling, a WebSocket connection remains open and allows the server to push updates as they occur, reducing latency and network overhead for live financial data.

> __Learning Objectives__
>
> By the end of this activity, you will be able to:
> * __Connect to a WebSocket API:__ Establish a connection to the Kraken WebSocket endpoint and send subscription messages to receive real-time cryptocurrency price feeds.
> * __Handle WebSocket messages:__ Parse incoming JSON messages and implement conditional logic to process different message types including status updates, heartbeats, and ticker data.
> * __Manage connection lifecycle:__ Control the WebSocket connection by monitoring message counts and gracefully closing the connection when criteria are met.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

### Constants
For this activity, we will connect to the Kraken exchange and monitor real-time price data for Bitcoin. We specify the WebSocket URL, the currency pair to track, and how many messages to receive before closing the connection.

In [ ]:
url = "wss://ws.kraken.com/v2"; # WebSocket URL for Kraken's public API
pair = "BTC/USD"; # Currency pair to monitor (Bitcoin to US Dollar)
number_of_messages = 2; # Number of messages to receive
const WS = HTTP.WebSockets; # Create alias for WebSockets module for convenience

___

## Task 1: Implement the WebSocket Listener
In this task, you will implement a WebSocket listener that connects to the Kraken API and receives real-time price quotes for the specified cryptocurrency pair. 

Let's start by setting up the subscription message to send to the WebSocket server. We save this message in a variable called `subscription_message::Dict{String, Any}`.

> __What's in a subscription message?__
>
> A subscription message is a JSON-formatted request that tells the WebSocket server which data streams you want to receive. The message contains a `method` field set to `"subscribe"` and a `params` field specifying the channel (e.g., `ticker` for price updates) and the symbol (e.g., `BTC/USD` for Bitcoin to US Dollar).


In [3]:
subscription_message = Dict(
    "method" => "subscribe",
    "params" => Dict("channel" => "ticker", "symbol" => ["$(pair)"]),
);

With the subscription message ready, we can implement the WebSocket listener to connect to the Kraken API and start receiving real-time price quotes.

> __WebSocket connection and message handling__
>
> [The `WS.open(...)` function (alias for `HTTP.WebSockets.open`)](https://juliaweb.github.io/HTTP.jl/stable/reference/#HTTP.open) establishes a WebSocket connection to the specified URL. The `do` block receives a `ws` handle representing the open connection. We then send the subscription message using `WS.send(ws, ...)` and iterate over incoming messages with a `for` loop. Each message arrives as either a `String` or `Vector{UInt8}`, which we normalize to a string and parse as JSON using `JSON.parse(...)`.

The code includes error handling to deal with malformed messages:

> __Error handling for JSON parsing__
>
> The `try-catch` block wraps `JSON.parse(s)` to catch any parsing errors. If a message cannot be parsed as valid JSON, the code logs a warning with the error details and the raw message, then continues to the next message using `continue`. This defensive programming pattern ensures that a single malformed message does not crash the entire WebSocket client.

This processing structure allows us to handle different message types:

> __Message type routing__
>
> Kraken sends several message types over the WebSocket: `status` messages confirm connection state, `heartbeat` messages indicate the connection is alive, and `ticker` messages with `type == "update"` contain the actual price data. We use conditional logic to route each message type appropriately. The ticker updates contain price information in the `msg["data"]` field. You can extract specific fields like the current price (`msg["data"][0]["last"]`), bid/ask prices, volume, and other market data from this nested structure.

Finally, we track the message count and close the connection when we reach the limit:

> __Connection lifecycle management__
>
> The `seen` variable increments with each message. When `seen >= max_messages`, we call `WS.close(ws)` to gracefully terminate the connection and exit the loop. This pattern ensures we collect a fixed number of updates and then clean up resources.

Try running the code to see the live ticker updates!

In [4]:
WS.open(url) do ws
    WS.send(ws, JSON.json(subscription_message))
    seen = 0; # count of messages seen
    max_messages = number_of_messages; # number of messages to receive (then we close the connection)

    for raw in ws
        s = raw isa AbstractString ? raw : String(raw) # raw can be String or Vector{UInt8}; normalize to String

        # ok: parse the JSON message. If something goes wrong, log a warning and skip to the next message
        msg = try
            JSON.parse(s)
        catch err
            @warn "failed to parse websocket payload" error=err raw=s
            continue
        end

        # Kraken sends a status update on connect; just ignore or print it
        if msg isa JSON.Object{String, Any}
            channel = get(msg, "channel", nothing) # get the channel field if it exists
            msg_type = get(msg, "type", nothing) # get the type field if it exists

            # here: we can handle different message types as desired
            if channel == "status"
                @info "status" msg # echo the status message
                continue
            elseif channel == "heartbeat"
                @info "heartbeat" msg # echo the heartbeat message
                continue
            elseif channel == "ticker" && msg_type == "update"
                @info "ticker update" msg
                
                # Your logic goes here!
                # process ticker data here if desired; Kraken packs updates in msg["data"]
                continue
            elseif channel == "subscribe"
                @info "subscription status" msg
                continue
            end
        end
        @info "unhandled payload" msg

        seen += 1 # update message count
        if seen >= max_messages
            @info "max message count reached; closing websocket" count=seen
            WS.close(ws) # close the WebSocket connection
            break
        end
    end
end

┌ Info: status
│   msg = JSON.Object{String, Any}("channel" => "status", "type" => "update", "data" => Any[JSON.Object{String, Any}("version" => "2.0.10", "system" => "online", "api_version" => "v2", "connection_id" => 9980070655971841378)])
└ @ Main /Users/jdv27/Desktop/julia_work/CHEME-140-eCornell-Repository/courses/CHEME-142/module-3/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X12sZmlsZQ==.jl:24
┌ Info: unhandled payload
│   msg = JSON.Object{String, Any}("method" => "subscribe", "result" => JSON.Object{String, Any}("channel" => "ticker", "event_trigger" => "trades", "snapshot" => true, "symbol" => "BTC/USD"), "success" => true, "time_in" => "2025-12-17T17:23:07.034333Z", "time_out" => "2025-12-17T17:23:07.034375Z")
└ @ Main /Users/jdv27/Desktop/julia_work/CHEME-140-eCornell-Repository/courses/CHEME-142/module-3/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X12sZmlsZQ==.jl:40
┌ Info: unhandled payload
│   msg = JSON.Object{String, Any}("channel" => "ticker", "type" => "sn

HTTP.Messages.Response:
"""
HTTP/1.1 101 Switching Protocols
Date: Wed, 17 Dec 2025 17:23:06 GMT
Connection: upgrade
upgrade: websocket
sec-websocket-accept: JcWl5rpCwB89koHHJ6udKVJ5Kh4=
CF-Cache-Status: BYPASS
Set-Cookie: ******
Vary: Accept-Encoding
Strict-Transport-Security: max-age=31536000; includeSubDomains; preload
X-Content-Type-Options: nosniff
Set-Cookie: ******
Server: cloudflare
CF-RAY: 9af81b37ad2ccb6b-EWR

"""

___

## Summary
This activity demonstrated how to establish a WebSocket connection to the Kraken cryptocurrency exchange and process real-time price updates.

> __Key Takeaways__
>
> * **WebSocket subscription pattern:** Connecting to a WebSocket API requires opening a connection, sending a subscription message specifying the desired data channel, and then iterating over incoming messages to process updates.
> * **Message type routing:** WebSocket servers often send multiple message types (status, heartbeat, data updates), which require conditional logic to handle appropriately based on message fields such as `channel` and `type`.
> * **Connection management:** Controlling the WebSocket lifecycle by tracking message counts and calling `close()` when done ensures clean resource management and prevents unnecessary data transfer.

This pattern applies to any WebSocket API that provides real-time data streams, from financial markets to social media feeds.
___